# Stage 7a — Error analysis on the saved cross-validated predictions (no training, CPU only)

**Why this notebook.** The final pipeline reached about **66 %** top-1 for one CNN and **69.7 %** for the best ensemble (300 signs, 5-fold signer-disjoint CV).
Before spending more compute, we want to know **where the remaining errors are**: which signs fail, which pairs are confused, whether errors depend on clip quality,
on one-hand versus two-hand signs, on the signer, and how trustworthy the model's confidence is. The answers decide what to do next
(for example: are face and body landmarks worth extracting, is a rejection threshold useful for an application, which signs need more data).

**What it uses:** `results/final_cv_logits.npz` (written by the final-pipeline notebook: the logits of every model on every cross-validation test clip, with clip ids and labels),
`processed/splits.csv`, and, optionally, `processed/sequences_subset_T64_robust3d.npz` (for the one-hand / two-hand analysis).
Every non-prolific clip appears **exactly once** as a test clip (1,932 clips), so these are honest out-of-signer predictions.

**What it does**
1. Rebuilds a per-clip table and checks that the accuracies match the final pipeline (a sanity check on the files).
2. Overall accuracy with a **signer-cluster bootstrap confidence interval**, top-k curve.
3. Per-sign accuracy, hardest and easiest signs.
4. The most confused pairs, and "sink" signs that attract wrong predictions.
5. Accuracy by clip length, missing frames, low coverage and one-hand vs two-hand.
6. Accuracy per signer.
7. Model agreement, calibration of the confidence, and a coverage-accuracy curve (what accuracy if the system only answers when it is confident).

**Caution:** a sign has only about 6 test clips in this analysis, so per-sign numbers are noisy. Look for patterns across many signs, not for single signs.

**How to run:** a new Colab notebook (CPU is enough, about 1-2 minutes), run the cells in order from Cell 2. If a cell fails, send me the last lines of the error.

In [ ]:
# [setup] Cell 2 — configuration
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/slovo_project')
except ImportError:                      # running outside Colab
    BASE = Path('./slovo_project')

PROC, FIG, RES = BASE / 'processed', BASE / 'figures', BASE / 'results'
for d in (FIG, RES):
    d.mkdir(parents=True, exist_ok=True)
N_FOLDS = 5
N_BOOT = 2000
warnings.filterwarnings('ignore')
print('Working directory:', BASE)

In [ ]:
# [load] Cell 3 — rebuild the per-clip table from the saved logits
Z = np.load(RES / 'final_cv_logits.npz', allow_pickle=False)
splits = pd.read_csv(PROC / 'splits.csv')
meta = splits[splits['in_subset']].reset_index(drop=True)
label_text = meta.drop_duplicates('label_id_subset').set_index('label_id_subset')['text'].to_dict()
info = meta.set_index('attachment_id')

model_keys = sorted({k.rsplit('_fold', 1)[0] for k in Z.files if k.endswith('_logits')})
print('models found:', model_keys)

def softmax(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z)
    return e / e.sum(1, keepdims=True)

ids = np.concatenate([Z[f'fold{k}_ids'] for k in range(N_FOLDS)])
y = np.concatenate([Z[f'fold{k}_y'] for k in range(N_FOLDS)]).astype(int)
fold = np.concatenate([np.full(len(Z[f'fold{k}_ids']), k) for k in range(N_FOLDS)])
P = {m: np.concatenate([softmax(Z[f'{m}_fold{k}_logits'].astype(np.float32)) for k in range(N_FOLDS)]) for m in model_keys}
N, K = P[model_keys[0]].shape
assert len(ids) == len(y) == N and len(set(ids)) == N, 'clip ids must be unique'

ENSEMBLE = model_keys                                      # all saved models; change to a subset if you want to compare
P_ens = np.mean([P[m] for m in ENSEMBLE], axis=0)

df = pd.DataFrame({'id': ids, 'fold': fold, 'y': y})
df = df.join(info[['user_id', 'n_frames', 'missing_rate', 'low_coverage', 'aspect']], on='id')
df['sign'] = df['y'].map(label_text)
df['pred'] = P_ens.argmax(1)
df['pred_sign'] = df['pred'].map(label_text)
df['conf'] = P_ens.max(1)
df['correct'] = (df['pred'] == df['y'])
df['rank'] = (P_ens > P_ens[np.arange(N), y][:, None]).sum(1)          # 0 = the true sign is the top-1 guess
for m in model_keys:
    df[f'correct_{m}'] = (P[m].argmax(1) == y)
print(f'{N} cross-validated test clips | {df["user_id"].nunique()} signers | {df["y"].nunique()} signs | {K} classes')
print(df.head(3).to_string())

In [ ]:
# Cell 4 — overall accuracy (sanity check against the final pipeline), signer-cluster bootstrap, top-k curve
print('top-1 per model and fold (compare with the final pipeline notebook):')
rows = {m: [df.loc[df.fold == k, f'correct_{m}'].mean() for k in range(N_FOLDS)] for m in model_keys}
rows['ensemble (all models)'] = [df.loc[df.fold == k, 'correct'].mean() for k in range(N_FOLDS)]
tab = pd.DataFrame(rows, index=[f'fold {k}' for k in range(N_FOLDS)]).T
tab['mean'] = tab.mean(1); tab['std'] = tab.iloc[:, :N_FOLDS].std(1, ddof=0)
print((tab * 100).round(1).to_string())

def cluster_bootstrap(correct, groups, n=N_BOOT, seed=0):
    """95 % interval of the accuracy when SIGNERS (not clips) are resampled: clips of one signer are correlated."""
    rng = np.random.default_rng(seed)
    uniq = np.unique(groups)
    idx_by = {g: np.where(groups == g)[0] for g in uniq}
    accs = [correct[np.concatenate([idx_by[g] for g in rng.choice(uniq, len(uniq), replace=True)])].mean() for _ in range(n)]
    return np.percentile(accs, [2.5, 97.5])

print('\nsigner-cluster bootstrap 95 % intervals (pooled accuracy over all test clips):')
for m in model_keys + ['ensemble']:
    c = df['correct'].to_numpy() if m == 'ensemble' else df[f'correct_{m}'].to_numpy()
    lo, hi = cluster_bootstrap(c, df['user_id'].to_numpy())
    print(f'  {m:10s} {c.mean():.1%}  [{lo:.1%}, {hi:.1%}]')

topk = [(df['rank'] < k).mean() for k in range(1, 11)]
print('\ntop-k accuracy of the ensemble: ' + ' | '.join(f'k={k}: {a:.1%}' for k, a in zip(range(1, 11), topk) if k in (1, 2, 3, 5, 10)))
plt.figure(figsize=(5, 3.2)); plt.plot(range(1, 11), topk, marker='o'); plt.xlabel('k'); plt.ylabel('top-k accuracy'); plt.grid(alpha=.3)
plt.title('The right sign is in the first k guesses'); plt.tight_layout(); plt.savefig(FIG / 'stage7a_topk.png', dpi=150); plt.show()

In [ ]:
# Cell 5 — per-sign accuracy: which signs are hard?
per_sign = df.groupby('sign').agg(n=('correct', 'size'), acc=('correct', 'mean'), top5=('rank', lambda r: (r < 5).mean()),
                                  label=('y', 'first')).sort_values('acc')
print(f'signs: {len(per_sign)} | clips per sign: median {int(per_sign.n.median())}, min {per_sign.n.min()}, max {per_sign.n.max()}')
print(f'macro (per-sign) accuracy {per_sign.acc.mean():.1%} vs overall {df.correct.mean():.1%}')
print(f'signs never recognized (accuracy 0 %): {(per_sign.acc == 0).sum()} | signs recognized at least 80 %: {(per_sign.acc >= 0.8).sum()}')
reliable = per_sign[per_sign.n >= 4]
print('\n15 hardest signs (at least 4 test clips):'); print((reliable.head(15)[['n', 'acc', 'top5']] * [1, 100, 100]).round(0).to_string())
print('\n10 easiest signs (at least 4 test clips):'); print((reliable.tail(10)[['n', 'acc', 'top5']] * [1, 100, 100]).round(0).to_string())

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].hist(per_sign.acc, bins=np.linspace(0, 1, 11), color='#4C78A8', edgecolor='w'); ax[0].set_xlabel('accuracy of a sign'); ax[0].set_ylabel('number of signs')
ax[0].set_title('Per-sign accuracy (about 6 clips per sign)')
ax[1].scatter(per_sign.n + np.random.default_rng(0).uniform(-.15, .15, len(per_sign)), per_sign.acc, s=10, alpha=.6)
ax[1].set_xlabel('test clips of the sign'); ax[1].set_ylabel('accuracy'); ax[1].set_title('Signs with few test clips are noisy')
plt.tight_layout(); plt.savefig(FIG / 'stage7a_per_sign.png', dpi=150); plt.show()

In [ ]:
# Cell 6 — the most confused sign pairs and the "sink" signs that attract wrong predictions
err = df[~df.correct]
pairs = err.groupby(['sign', 'pred_sign']).size().rename('errors').reset_index()
pairs['clips_of_true_sign'] = pairs['sign'].map(per_sign['n'])
pairs['share'] = pairs['errors'] / pairs['clips_of_true_sign']
rev = {(a, b): c for a, b, c in zip(pairs['sign'], pairs['pred_sign'], pairs['errors'])}
pairs['reverse_errors'] = [rev.get((b, a), 0) for a, b in zip(pairs['sign'], pairs['pred_sign'])]
top_pairs = pairs.sort_values(['errors', 'share'], ascending=False).head(20)
print(f'errors: {len(err)} of {N} clips ({len(err) / N:.1%}); distinct wrong pairs: {len(pairs)}; pairs with 2 or more errors: {(pairs.errors >= 2).sum()}')
print('\n20 most frequent confusions (true sign -> predicted sign):')
print(top_pairs[['sign', 'pred_sign', 'errors', 'clips_of_true_sign', 'reverse_errors']].to_string(index=False))
sym = ((pairs.errors >= 2) & (pairs.reverse_errors >= 2)).sum() / max((pairs.errors >= 2).sum(), 1)
print(f'\nshare of repeated confusions (2 or more) that are also confused in the opposite direction: {sym:.0%}')

sinks = err['pred_sign'].value_counts().head(10)
print('\nsigns most often predicted by mistake (sinks):'); print(sinks.to_string())
print(f'the 10 biggest sinks account for {sinks.sum() / len(err):.1%} of all errors (random would be about {10 / K:.1%})')

plt.figure(figsize=(7, 4.2))
lab = [f'{a} -> {b}' for a, b in zip(top_pairs.sign.head(12), top_pairs.pred_sign.head(12))]
plt.barh(range(len(lab))[::-1], top_pairs.errors.head(12), color='#B279A2'); plt.yticks(range(len(lab))[::-1], lab, fontsize=8)
plt.xlabel('number of wrong predictions'); plt.title('Most frequent confusions'); plt.tight_layout()
plt.savefig(FIG / 'stage7a_confusions.png', dpi=150); plt.show()

In [ ]:
# Cell 7 — does accuracy depend on clip quality or on one-hand vs two-hand signs?
def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); m = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return ((c - m) / d, (c + m) / d)

def by_group(col, bins=None, labels=None):
    g = pd.cut(df[col], bins=bins, labels=labels, include_lowest=True) if bins is not None else df[col]
    t = df.groupby(g).agg(n=('correct', 'size'), acc=('correct', 'mean'), k=('correct', 'sum'))
    t[['lo', 'hi']] = [wilson(k, n) for k, n in zip(t.k, t.n)]
    return t

groups = {}
q = df['n_frames'].quantile([0, .25, .5, .75, 1]).to_numpy()
groups['clip length (frames)'] = by_group('n_frames', bins=q, labels=[f'{int(q[i])}-{int(q[i + 1])}' for i in range(4)])
groups['frames missing from the landmarks'] = by_group('missing_rate', bins=[0, 0.02, 0.05, 0.1, 0.2, 1.0], labels=['<2%', '2-5%', '5-10%', '10-20%', '>20%'])
groups['low coverage (>50% missing)'] = by_group('low_coverage')

SEQF = PROC / 'sequences_subset_T64_robust3d.npz'
if SEQF.exists():
    zs = np.load(SEQF, allow_pickle=False)
    row_of = {c: i for i, c in enumerate(zs['ids'])}
    S = zs['S']
    vis0 = np.array([S[row_of[c], :, 130].astype(np.float32).mean() for c in df['id']])
    vis1 = np.array([S[row_of[c], :, 131].astype(np.float32).mean() for c in df['id']])
    df['hands'] = np.where((vis0 > 0.5) & (vis1 > 0.5), 'two hands', np.where((vis0 > 0.5) | (vis1 > 0.5), 'one hand', 'no hand'))
    groups['hands visible in most frames'] = by_group('hands')
else:
    print('(sequences_subset_T64_robust3d.npz not found: one-hand / two-hand analysis skipped)')

for name, t in groups.items():
    print(f'\n{name}'); print(pd.DataFrame({'clips': t.n, 'accuracy %': (t.acc * 100).round(1), '95% interval': [f'[{lo:.0%}, {hi:.0%}]' for lo, hi in zip(t.lo, t.hi)]}).to_string())

fig, ax = plt.subplots(1, len(groups), figsize=(4 * len(groups), 3.2))
ax = np.atleast_1d(ax)
for a, (name, t) in zip(ax, groups.items()):
    a.bar(range(len(t)), t.acc, yerr=[t.acc - t.lo, t.hi - t.acc], capsize=3, color='#4C78A8'); a.set_xticks(range(len(t)))
    a.set_xticklabels([str(i) for i in t.index], rotation=20, fontsize=8); a.set_title(name, fontsize=9); a.axhline(df.correct.mean(), color='k', ls='--', lw=.8)
    a.set_ylim(0, 1)
plt.tight_layout(); plt.savefig(FIG / 'stage7a_by_clip_property.png', dpi=150); plt.show()

In [ ]:
# Cell 8 — accuracy per signer (the test signers are the non-prolific ones)
per_signer = df.groupby('user_id').agg(n=('correct', 'size'), acc=('correct', 'mean')).sort_values('n', ascending=False)
big = per_signer[per_signer.n >= 15]
print(f'{len(per_signer)} test signers; {len(big)} signers have 15 or more test clips (they supply {big.n.sum() / N:.0%} of the clips)')
print('signers with the most test clips:'); print(big.head(10).assign(acc=lambda d: (d.acc * 100).round(1)).to_string())
if len(big) > 1:
    print(f'\nspread of accuracy across these signers: min {big.acc.min():.1%}, max {big.acc.max():.1%}, std {big.acc.std():.1%} '
          f'(pure sampling noise alone would give about {np.sqrt(df.correct.mean() * (1 - df.correct.mean()) / big.n.mean()):.1%})')
plt.figure(figsize=(5.5, 3.4)); plt.scatter(per_signer.n, per_signer.acc, s=14, alpha=.7); plt.xscale('log'); plt.xlabel('test clips of the signer (log)')
plt.ylabel('accuracy'); plt.title('Accuracy per signer'); plt.axhline(df.correct.mean(), color='k', ls='--', lw=.8); plt.tight_layout()
plt.savefig(FIG / 'stage7a_per_signer.png', dpi=150); plt.show()

In [ ]:
# Cell 9 — model agreement, calibration of the confidence, and the coverage-accuracy curve
cols = [f'correct_{m}' for m in model_keys]
if len(model_keys) >= 2:
    a, b = model_keys[0], model_keys[-1]
    agree = (P[a].argmax(1) == P[b].argmax(1))
    print(f'{a} and {b} give the same answer on {agree.mean():.0%} of the clips; ensemble accuracy {df.correct[agree].mean():.1%} when they agree, '
          f'{df.correct[~agree].mean():.1%} when they disagree')

bins = np.linspace(0, 1, 11)
cb = pd.cut(df.conf, bins=bins, include_lowest=True)
cal = df.groupby(cb).agg(n=('correct', 'size'), acc=('correct', 'mean'), conf=('conf', 'mean'))
print('\nreliability of the ensemble confidence (probability of its top guess):')
print(pd.DataFrame({'clips': cal.n, 'mean confidence': cal.conf.round(2), 'accuracy': cal.acc.round(2)}).dropna().to_string())

order = np.argsort(-df.conf.to_numpy())
cum_acc = np.cumsum(df.correct.to_numpy()[order]) / np.arange(1, N + 1)
print('\nif the system only answers when it is confident (coverage = share of clips answered):')
for cov in (1.0, 0.9, 0.8, 0.7, 0.6, 0.5):
    i = int(cov * N) - 1
    print(f'  coverage {cov:.0%}: accuracy {cum_acc[i]:.1%} (confidence threshold {df.conf.to_numpy()[order][i]:.2f})')

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
cc = cal.dropna()
ax[0].plot([0, 1], [0, 1], 'k--', lw=.8); ax[0].plot(cc.conf, cc.acc, marker='o'); ax[0].set_xlabel('mean confidence'); ax[0].set_ylabel('accuracy'); ax[0].set_title('Reliability diagram')
ax[1].plot(np.arange(1, N + 1) / N, cum_acc); ax[1].set_xlabel('coverage (share of clips answered)'); ax[1].set_ylabel('accuracy of the answered clips'); ax[1].set_title('Answer only when confident')
ax[1].grid(alpha=.3); plt.tight_layout(); plt.savefig(FIG / 'stage7a_confidence.png', dpi=150); plt.show()

In [ ]:
# Cell 10 — save the tables and a short summary
df.drop(columns=[c for c in df.columns if c.startswith('correct_')]).to_csv(RES / 'stage7a_clip_predictions.csv', index=False)
per_sign.reset_index().to_csv(RES / 'stage7a_per_sign.csv', index=False)
pairs.sort_values('errors', ascending=False).to_csv(RES / 'stage7a_confusion_pairs.csv', index=False)
summary = {'clips': int(N), 'signers': int(df.user_id.nunique()), 'ensemble_models': ENSEMBLE, 'ensemble_top1': float(df.correct.mean()),
           'macro_sign_accuracy': float(per_sign.acc.mean()), 'signs_never_recognized': int((per_sign.acc == 0).sum()),
           'top5': float((df['rank'] < 5).mean()), 'errors': int(len(err)),
           'ten_biggest_sinks_share_of_errors': float(sinks.sum() / len(err)),
           'repeated_confusions_also_reverse': float(sym)}
json.dump(summary, open(RES / 'stage7a_summary.json', 'w'), indent=2)
print(json.dumps(summary, indent=2))
print('Saved the tables and figures in', RES, 'and', FIG)

## Stage 7a checklist

- [ ] Cell 3 found the models and 1,932 unique test clips
- [ ] Cell 4: per-fold accuracies match the final pipeline (for example the two-CNN + GRU ensemble about 69.7 %), and the bootstrap interval is printed
- [ ] Cells 5 and 6: hardest signs, most confused pairs, sinks
- [ ] Cell 7: accuracy by clip length, missing frames, low coverage, hands
- [ ] Cells 8 and 9: per-signer accuracy, confidence reliability, coverage-accuracy curve
- [ ] `results/stage7a_*.csv`, `results/stage7a_summary.json` and the figures exist

**Send back:** the printed outputs of Cells 3 to 10 and the figures `stage7a_confusions.png`, `stage7a_by_clip_property.png` and `stage7a_confidence.png`.
If a cell fails, send me the last lines of the error.